In [1]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated     

In [2]:
load_dotenv(override=True) 

True

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]   

In [4]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1"
)

In [5]:
def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}

In [6]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()


In [7]:
result = graph.invoke({
    "messages": [HumanMessage(content="Who is the US President")]
})

In [8]:
for msg in result["messages"]:
    if hasattr(msg, 'content'):
        role = "Request" if isinstance(msg, HumanMessage) else "AI Response"
        print(f"{role}: {msg.content}\n")

Request: Who is the US President

AI Response: As of my last knowledge update in October 2023, the President of the United States is Joe Biden. He was inaugurated on January 20, 2021. If you need the most current information, please verify with a reliable news source.



Tools

In [9]:
from langchain_core.tools import tool

In [11]:
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool when you need to perform calculations.
    
    Args:
        expression: A math expression like '2 + 2' or '100 * 0.15'
    """
    try:
        result = eval(expression)          
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

In [14]:
@tool
def search_web(query: str) -> str:
    """
    Search the web for current information.
    Use when asked about recent events, news, or facts you're unsure about.
    
    Args:
        query: The search query (e.g., 'latest AI news 2025')
    """
    # Mock implementation — replace with real search in production
    results = {
        "weather today": "Sunny, 25°C in most areas",
        "latest ai news": "OpenAI released GPT-5, Google launched Gemini 3.0",
        "stock market": "S&P 500 up 1.2% today, tech sector leading"
    }
    for key, value in results.items():
        if key in query.lower():
            return value
    return f"Search results for '{query}': No specific results found. This is a mock search."

In [15]:
@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    Use when asked about weather conditions.
    
    Args:
        city: The city name (e.g., 'London', 'Tokyo', 'New York')
    """
    weather_data = {
        "london": "Cloudy, 15°C, 60% humidity",
        "tokyo": "Sunny, 28°C, 45% humidity",
        "new york": "Rainy, 18°C, 80% humidity",
        "paris": "Partly cloudy, 20°C, 55% humidity",
    }
    return weather_data.get(city.lower(), f"Weather data unavailable for {city}")

In [16]:
tools = [calculator, search_web, get_weather]

In [17]:
llm_with_tools = llm.bind_tools(tools)